## SPARK SQL DEMO

In [ ]:
#pip install pyspark

## Creating a SparkSession

- A **SparkSession** object is the entry point to Spark.  
- It **wraps the Driver program** (the "brain" of your Spark app).  
- Every Spark command you run (like `spark.read.csv(...)` or `df.groupBy(...)`) goes through this `spark` object.  

### Example
```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()


In [1]:
# --- Setup SparkSession (local mode) ---
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Beginner-DE-Demo")
    .master("local[*]")         # use all local cores
    .getOrCreate()
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/11 19:06:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/09/11 19:06:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
spark

## 1. Read CSV into a DataFrame

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

# Path to the CSV DATASET
csv_path = "Data/retail_events.csv"

In [4]:
#create a dataframe, similar to pandas but we have more POWER
df_raw = (
    spark.read
    .option("header", True)
    .csv(csv_path)
)


In [5]:
#create a clone of raw dataframe
df = df_raw

In [6]:
df.show()

+-------------------+-------+----------+-------+-------+------+------+--------+----------+--------------------+
|         event_time|user_id|product_id|country| device|source| price|quantity|promo_code|               email|
+-------------------+-------+----------+-------+-------+------+------+--------+----------+--------------------+
|2025-08-10T15:06:00|   1007|       204|     US|android| email| 66.92|       1|      NULL|user1007@example.com|
|2025-08-10T12:10:00|   1006|       200|     DE|    web|   ads| 28.79|       2|  SUMMER25|user1006@example.com|
|2025-08-10T10:24:00|   1007|       201|     US|    web|direct| 70.94|       2|      NULL| USER1007@EXAMPLE...|
|2025-08-10T15:21:00|   1005|       203|     US|    ios|   seo| 66.87|       2|      NULL|user1005@example.com|
|2025-08-10T09:17:00|   1003|       204|     CA|    ios|   ads| 27.83|       1|      NULL|user1003@example.com|
|2025-08-10T09:36:00|   1007|       204|     GB|android|   seo|  79.8|       3|      NULL|user1007@examp

# 2. SPARK SQL

In [7]:
# Register as a temp view for SQL aka a sql TABLE!
df.createOrReplaceTempView("retail_raw")

In [8]:
spark.sql("SELECT * FROM retail_raw")

DataFrame[event_time: string, user_id: string, product_id: string, country: string, device: string, source: string, price: string, quantity: string, promo_code: string, email: string]

In [9]:
spark.sql("SELECT * FROM retail_raw").show()

+-------------------+-------+----------+-------+-------+------+------+--------+----------+--------------------+
|         event_time|user_id|product_id|country| device|source| price|quantity|promo_code|               email|
+-------------------+-------+----------+-------+-------+------+------+--------+----------+--------------------+
|2025-08-10T15:06:00|   1007|       204|     US|android| email| 66.92|       1|      NULL|user1007@example.com|
|2025-08-10T12:10:00|   1006|       200|     DE|    web|   ads| 28.79|       2|  SUMMER25|user1006@example.com|
|2025-08-10T10:24:00|   1007|       201|     US|    web|direct| 70.94|       2|      NULL| USER1007@EXAMPLE...|
|2025-08-10T15:21:00|   1005|       203|     US|    ios|   seo| 66.87|       2|      NULL|user1005@example.com|
|2025-08-10T09:17:00|   1003|       204|     CA|    ios|   ads| 27.83|       1|      NULL|user1003@example.com|
|2025-08-10T09:36:00|   1007|       204|     GB|android|   seo|  79.8|       3|      NULL|user1007@examp

In [10]:
spark.sql("SELECT COUNT(*) FROM retail_raw").show()

+--------+
|count(1)|
+--------+
|      41|
+--------+



In [11]:
spark.sql("SELECT * FROM retail_raw").show(10, truncate=False)

+-------------------+-------+----------+-------+-------+------+------+--------+----------+----------------------+
|event_time         |user_id|product_id|country|device |source|price |quantity|promo_code|email                 |
+-------------------+-------+----------+-------+-------+------+------+--------+----------+----------------------+
|2025-08-10T15:06:00|1007   |204       |US     |android|email |66.92 |1       |NULL      |user1007@example.com  |
|2025-08-10T12:10:00|1006   |200       |DE     |web    |ads   |28.79 |2       |SUMMER25  |user1006@example.com  |
|2025-08-10T10:24:00|1007   |201       |US     |web    |direct|70.94 |2       |NULL      | USER1007@EXAMPLE.COM |
|2025-08-10T15:21:00|1005   |203       |US     |ios    |seo   |66.87 |2       |NULL      |user1005@example.com  |
|2025-08-10T09:17:00|1003   |204       |CA     |ios    |ads   |27.83 |1       |NULL      |user1003@example.com  |
|2025-08-10T09:36:00|1007   |204       |GB     |android|seo   |79.8  |3       |NULL     

# 3. Basic Cleaning
- Cast columns (e.g., string → timestamp)
- Normalize strings (trim, lowercase/uppercase)
- Handle nulls and empty values
- Drop duplicates
- Filter out invalid data (e.g., quantity > 0)

### Every new transformation we create a new TEMP TABLE 

In [12]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW retail_clean AS
SELECT
  NULLIF(TRIM(country), '')           AS country,
  LOWER(TRIM(device))                 AS device,
  LOWER(TRIM(source))                 AS source,
  NULLIF(UPPER(TRIM(promo_code)), '') AS promo_code,
  LOWER(TRIM(email))                  AS email,
  event_time,
  user_id,
  product_id,
  price,
  quantity
FROM retail_raw
""")


DataFrame[]

In [13]:
spark.sql("SELECT * FROM retail_clean").show(10, truncate=False)

+-------+-------+------+----------+--------------------+-------------------+-------+----------+------+--------+
|country|device |source|promo_code|email               |event_time         |user_id|product_id|price |quantity|
+-------+-------+------+----------+--------------------+-------------------+-------+----------+------+--------+
|US     |android|email |NULL      |user1007@example.com|2025-08-10T15:06:00|1007   |204       |66.92 |1       |
|DE     |web    |ads   |SUMMER25  |user1006@example.com|2025-08-10T12:10:00|1006   |200       |28.79 |2       |
|US     |web    |direct|NULL      |user1007@example.com|2025-08-10T10:24:00|1007   |201       |70.94 |2       |
|US     |ios    |seo   |NULL      |user1005@example.com|2025-08-10T15:21:00|1005   |203       |66.87 |2       |
|CA     |ios    |ads   |NULL      |user1003@example.com|2025-08-10T09:17:00|1003   |204       |27.83 |1       |
|GB     |android|seo   |NULL      |user1007@example.com|2025-08-10T09:36:00|1007   |204       |79.8  |3 

In [14]:
# Sanity checks ---

# Sample rows
spark.sql("SELECT country, device, source, promo_code, email FROM retail_clean LIMIT 10").show(truncate=False)


+-------+-------+------+----------+--------------------+
|country|device |source|promo_code|email               |
+-------+-------+------+----------+--------------------+
|US     |android|email |NULL      |user1007@example.com|
|DE     |web    |ads   |SUMMER25  |user1006@example.com|
|US     |web    |direct|NULL      |user1007@example.com|
|US     |ios    |seo   |NULL      |user1005@example.com|
|CA     |ios    |ads   |NULL      |user1003@example.com|
|GB     |android|seo   |NULL      |user1007@example.com|
|CA     |android|direct|SUMMER25  |user1006@example.com|
|IN     |ios    |email |NULL      |user1000@example.com|
|IN     |web    |email |SUMMER25  |user1008@example.com|
|DE     |ios    |direct|SUMMER25  |user1007@example.com|
+-------+-------+------+----------+--------------------+



## Null counts (per column)

In [15]:

spark.sql("""
SELECT
  SUM(CASE WHEN country   IS NULL THEN 1 ELSE 0 END) AS country_nulls,
  SUM(CASE WHEN device    IS NULL THEN 1 ELSE 0 END) AS device_nulls,
  SUM(CASE WHEN source    IS NULL THEN 1 ELSE 0 END) AS source_nulls,
  SUM(CASE WHEN promo_code IS NULL THEN 1 ELSE 0 END) AS promo_code_nulls,
  SUM(CASE WHEN email     IS NULL THEN 1 ELSE 0 END) AS email_nulls
FROM retail_clean
""").show(truncate=False)

+-------------+------------+------------+----------------+-----------+
|country_nulls|device_nulls|source_nulls|promo_code_nulls|email_nulls|
+-------------+------------+------------+----------------+-----------+
|1            |0           |0           |23              |0          |
+-------------+------------+------------+----------------+-----------+



## Distinct counts (non-NULL)

In [16]:

spark.sql("""
SELECT
  COUNT(DISTINCT country)    AS country_distinct,
  COUNT(DISTINCT device)     AS device_distinct,
  COUNT(DISTINCT source)     AS source_distinct,
  COUNT(DISTINCT promo_code) AS promo_code_distinct,
  COUNT(DISTINCT email)      AS email_distinct
FROM retail_clean
""").show(truncate=False)

+----------------+---------------+---------------+-------------------+--------------+
|country_distinct|device_distinct|source_distinct|promo_code_distinct|email_distinct|
+----------------+---------------+---------------+-------------------+--------------+
|5               |3              |4              |2                  |9             |
+----------------+---------------+---------------+-------------------+--------------+



## Drop exact duplicate rows (on all columns)

In [17]:

spark.sql("""
CREATE OR REPLACE TEMP VIEW retail_clean2 AS
SELECT DISTINCT *
FROM retail_clean
""")

DataFrame[]

In [19]:
spark.sql("select * from retail_clean2").show()

+-------+-------+------+----------+--------------------+-------------------+-------+----------+------+--------+
|country| device|source|promo_code|               email|         event_time|user_id|product_id| price|quantity|
+-------+-------+------+----------+--------------------+-------------------+-------+----------+------+--------+
|     CA|android|direct|  SUMMER25|user1006@example.com|2025-08-10T13:39:00|   1006|       203|113.17|       1|
|     US|android|   ads|  SUMMER25|user1000@example.com|2025-08-10T15:54:00|   1000|       203| 32.85|       2|
|     GB|    web|direct|  SUMMER25|user1003@example.com|2025-08-10T09:07:00|   1003|       205| 65.57|       3|
|     DE|    web|   ads|  SUMMER25|user1006@example.com|2025-08-10T12:10:00|   1006|       200| 28.79|       2|
|     GB|    ios|direct|  SUMMER25|user1003@example.com|2025-08-10T18:12:00|   1003|       202|  9.22|       1|
|     US|    ios|   seo|  SUMMER25|user1000@example.com|2025-08-10T12:59:00|   1000|       201| 100.1|  

## Simple data quality filters (business rules)

In [21]:

#    - quantity should be >= 1 (remove zero or negative)
#    - price should be > 0
spark.sql("""
CREATE OR REPLACE TEMP VIEW retail_clean3 AS
SELECT *
FROM retail_clean2
WHERE CAST(quantity AS INT) >= 1
  AND CAST(price AS DOUBLE) > 0
""")

DataFrame[]

## Add agg columns

In [22]:

spark.sql("""
CREATE OR REPLACE TEMP VIEW retail_clean4 AS
SELECT
    *,
    (CAST(price AS DOUBLE) * CAST(quantity AS INT)) AS line_amount,   -- derived column
    TO_DATE(event_time) AS event_date                                 -- extract date
FROM retail_clean3
""")

DataFrame[]

## Calculate Revenue by country

In [23]:
# Run SQL query
rev_by_country = spark.sql("""
SELECT
    country,
    COUNT(*) AS rows,                      -- total rows (includes NULLs in other cols)
    SUM(line_amount) AS revenue,           -- total revenue
    COUNT(DISTINCT user_id) AS unique_users -- distinct users
FROM retail_clean4
GROUP BY country
ORDER BY revenue DESC
""")

rev_by_country.show(truncate=False)

+-------+----+-----------------+------------+
|country|rows|revenue          |unique_users|
+-------+----+-----------------+------------+
|US     |14  |1683.23          |9           |
|GB     |5   |628.9499999999999|3           |
|CA     |6   |615.32           |5           |
|DE     |3   |473.64           |2           |
|IN     |7   |409.56           |5           |
|NULL   |1   |50.2             |1           |
+-------+----+-----------------+------------+



## Calculate Top products by revenue

In [24]:

top_products = spark.sql("""
SELECT
    product_id,
    SUM(quantity)      AS units_sold,
    SUM(line_amount)   AS revenue
FROM retail_clean4
GROUP BY product_id
ORDER BY revenue DESC
""")

top_products.show(truncate=False)



+----------+----------+-------+
|product_id|units_sold|revenue|
+----------+----------+-------+
|201       |18.0      |998.74 |
|204       |11.0      |941.95 |
|203       |16.0      |752.49 |
|205       |8.0       |728.62 |
|200       |6.0       |303.86 |
|202       |4.0       |135.24 |
+----------+----------+-------+



## Daily revenue

In [25]:

daily_rev = spark.sql("""
SELECT
    event_date,
    SUM(line_amount)            AS revenue,
    COUNT(DISTINCT user_id)     AS unique_users
FROM retail_clean4
GROUP BY event_date
ORDER BY event_date
""")

daily_rev.show(truncate=False)

+----------+------------------+------------+
|event_date|revenue           |unique_users|
+----------+------------------+------------+
|2025-08-10|3860.8999999999996|9           |
+----------+------------------+------------+



## 5. Write Cleaned Data

In [26]:
# Choose output folders (local paths for notebook demo)
out_base = "Data/retail_events_out"

In [27]:
# Suppose you already have a cleaned view
# e.g., CREATE OR REPLACE TEMP VIEW retail_final AS SELECT ... FROM retail_raw

# 1) Run SQL and capture the DataFrame
df_clean = spark.sql("SELECT * FROM retail_clean4")



In [28]:
# 2) Write as a single CSV file with header
(
    df_clean.coalesce(1)              # force 1 output file
            .write
            .mode("overwrite")
            .option("header", True)
            .csv(f"{out_base}/spark_sql_csv_cleaned")
)


In [29]:
# Write Parquet (good default for analytics)
(
    df_clean.write
      .mode("overwrite")
      .parquet(f"{out_base}/spark_sql_parquet_cleaned")
)

print("Wrote cleaned CSV to:", f"{out_base}/csv_cleaned")
print("Wrote cleaned Parquet to:", f"{out_base}/parquet_cleaned")

[Stage 52:>                                                         (0 + 1) / 1]

Wrote cleaned CSV to: Data/retail_events_out/csv_cleaned
Wrote cleaned Parquet to: Data/retail_events_out/parquet_cleaned
